<a href="https://colab.research.google.com/github/kaguradance/AI/blob/main/train_and_preprocess.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# New Section

In [2]:
!pip install pythainlp emoji beautifulsoup4 xgboost joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.3/19.3 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 19.0 MB/s eta 0:00:00


In [3]:
import re
import json
import pandas as pd
import joblib
import unicodedata
from bs4 import BeautifulSoup
import emoji

# NLP Libraries
from pythainlp.tokenize import word_tokenize
from pythainlp.corpus import thai_stopwords

# Sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from scipy.sparse import hstack

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# XGBoost
from xgboost import XGBClassifier

In [7]:
thai_stops = thai_stopwords()

def clean_text(text):
    text = str(text)

    text = unicodedata.normalize("NFC", text)
    text = BeautifulSoup(text, "html.parser").get_text(separator=" ")

    # Emoji → text
    text = emoji.demojize(text)
    text = text.replace(":", " ")

    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"#\w+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"photo via [a-zA-Z0-9 ]+", "", text, flags=re.IGNORECASE)

    text = re.sub(r"[^\u0E00-\u0E7Fa-zA-Z0-9 ]", " ", text)

    tokens = word_tokenize(text, engine="newmm", keep_whitespace=False)
    tokens = [w for w in tokens if w not in thai_stops and len(w) > 1]

    return " ".join(tokens)

In [5]:
from google.colab import files
uploaded = files.upload()

Saving train_sentiment.json to train_sentiment.json


In [8]:
with open("train_sentiment.json", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)
df["text"] = df["text"].apply(clean_text)

X_text = df["text"]
y = df["sentiment"]

In [9]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [10]:
word_vec = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=50000,
    sublinear_tf=True
)

char_vec = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    max_features=30000
)

Xw_train = word_vec.fit_transform(X_train_text)
Xc_train = char_vec.fit_transform(X_train_text)
X_train = hstack([Xw_train, Xc_train])

Xw_test = word_vec.transform(X_test_text)
Xc_test = char_vec.transform(X_test_text)
X_test = hstack([Xw_test, Xc_test])

In [12]:
svm = LinearSVC(C=1.0, class_weight="balanced", random_state=42)
lr = LogisticRegression(max_iter=1000, class_weight="balanced", C=2.0)

xgb = XGBClassifier(
    eval_metric="logloss",
    random_state=42,
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1
)

stack_model = StackingClassifier(
    estimators=[
        ("svm", svm),
        ("lr", lr),
        ("xgb", xgb)
    ],
    final_estimator=LogisticRegression(),
    cv=5,
    n_jobs=-1
)

In [13]:
print("🚀 Training Model...")
stack_model.fit(X_train, y_train)

y_pred = stack_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

🚀 Training Model...


KeyboardInterrupt: 

In [ ]:
def plot_confusion_matrix(
    y_true,
    y_pred,
    model,
    title,
    normalize=True,
    figsize=(6, 5),
    cmap="Blues",
    save_path=None
):

    labels = model.classes_

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=labels,
        normalize="true" if normalize else None
    )

    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(cm, cmap=cmap)
    fig.colorbar(im, ax=ax)

    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels)
    ax.set_yticklabels(labels)

    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_title(title)

    fmt = ".2f" if normalize else "d"
    thresh = cm.max() / 2.0

    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(
                j,
                i,
                format(cm[i, j], fmt),
                ha="center",
                va="center",
                color="white" if cm[i, j] > thresh else "black"
            )

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

    plt.show()

plot_confusion_matrix(
    y_valid,
    y_valid_pred,
    model=stack_model,
    title="Normalized Confusion Matrix (Validation)",
    normalize=True,
    save_path="confusion_matrix_validation.png"
)

plot_confusion_matrix(
    y_test,
    y_test_pred,
    model=stack_model,
    title="Normalized Confusion Matrix (Test)",
    normalize=True,
    save_path="confusion_matrix_test.png"
)

In [ ]:
joblib.dump({
    "model": stack_model,
    "word_vec": word_vec,
    "char_vec": char_vec
}, "sentiment_bundle3.pkl")

print("✅ Training completed and all components saved")

✅ Training completed and all components saved


In [ ]:
files.download("sentiment_bundle3.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>